In [ ]:
import tensorflow as tf
import numpy as np
import scipy.io
from scipy.interpolate import griddata
import matplotlib.pyplot as plt
import time
from pyDOE import lhs
import scipy.optimize as sopt
import os

# Print versions for verification
print("TensorFlow version:", tf.__version__)
print("Numpy version:", np.__version__)
print("scipy version:", scipy.__version__)
print("matplotlib version:", plt.matplotlib.__version__)

# Set seeds for reproducibility
np.random.seed(1234)
tf.random.set_seed(1234)


class PhysicsInformedNN(tf.keras.Model):
    def __init__(self, X_data, u_v, X_f, lb, ub,
                 C10, C01, C20, C11, C02, g_x, g_y, layers, lambda_reg=1e-8):
        super().__init__()

        self.lambda_reg = tf.constant(lambda_reg, dtype=tf.float32)
        # Domain bounds and forcing
        self.lb = tf.constant(lb, dtype=tf.float32)
        self.ub = tf.constant(ub, dtype=tf.float32)
        self.g_x = tf.constant(g_x, dtype=tf.float32)
        self.g_y = tf.constant(g_y, dtype=tf.float32)
        # Material parameters
        self.C10 = tf.constant(C10, dtype=tf.float32)
        self.C01 = tf.constant(C01, dtype=tf.float32)
        self.C20 = tf.constant(C20, dtype=tf.float32)
        self.C11 = tf.constant(C11, dtype=tf.float32)
        self.C02 = tf.constant(C02, dtype=tf.float32)
        # Supervised data
        self.x = tf.convert_to_tensor(X_data[:,0:1], dtype=tf.float32)
        self.y = tf.convert_to_tensor(X_data[:,1:2], dtype=tf.float32)
        self.p = tf.convert_to_tensor(X_data[:,2:3], dtype=tf.float32)
        self.u = tf.convert_to_tensor(u_v[:,0:1], dtype=tf.float32)
        self.v = tf.convert_to_tensor(u_v[:,1:2], dtype=tf.float32)
        # Max and Min values
        self.u_min, self.u_max = tf.reduce_min(self.u), tf.reduce_max(self.u)
        self.v_min, self.v_max = tf.reduce_min(self.v), tf.reduce_max(self.v)
        self.u = 2*(self.u - self.u_min)/(self.u_max - self.u_min) - 1
        self.v = 2*(self.v - self.v_min)/(self.v_max - self.v_min) - 1
        # Collocation points
        self.x_f = tf.convert_to_tensor(X_f[:,0:1], dtype=tf.float32)
        self.y_f = tf.convert_to_tensor(X_f[:,1:2], dtype=tf.float32)
        self.p_f = tf.convert_to_tensor(X_f[:,2:3], dtype=tf.float32)
        # Build NN
        self.nn_weights, self.nn_biases = self.initialize_NN(layers)
        # Optimizer and history
        self.optimizer_Adam = tf.keras.optimizers.Adam(1e-3)
        self.data_history    = []
        self.physics_history = []
        self.total_history   = []

        # will be set once Adam is done:
        self.adam_steps = 0


    def initialize_NN(self, layers):
        Wb = []
        for i in range(len(layers)-1):
            in_dim, out_dim = layers[i], layers[i+1]
            std = np.sqrt(2/(in_dim+out_dim))
            W = tf.Variable(tf.random.truncated_normal([in_dim,out_dim],stddev=std), trainable=True)
            b = tf.Variable(tf.zeros([1,out_dim]), trainable=True)
            Wb.append((W,b))
        weights, biases = zip(*Wb)
        return list(weights), list(biases)

    def neural_net(self, X):
        H = 2*(X - self.lb)/(self.ub - self.lb) - 1
        for W,b in zip(self.nn_weights[:-1], self.nn_biases[:-1]):
            H = tf.tanh(tf.matmul(H,W) + b)
        W, b = self.nn_weights[-1], self.nn_biases[-1]
        return tf.matmul(H,W) + b

    @tf.function
    def net_data(self, x, y, p):
        uv = self.neural_net(tf.concat([x,y,p],1))
        return uv[:,0:1], uv[:,1:2]

    @tf.function
    def net_pde(self, x, y, p):
        with tf.GradientTape(persistent=True) as tape:
            tape.watch([x,y,p])
            
            uv = self.neural_net(tf.concat([x,y,p],1))
            u_pred, v_pred = uv[:,0:1], uv[:,1:2]
            ux = tape.gradient(u_pred, x); uy = tape.gradient(u_pred, y)
            vx = tape.gradient(v_pred, x); vy = tape.gradient(v_pred, y)
            F11, F12 = ux+1, uy
            F21, F22 = vx, vy+1
            F = tf.reshape(tf.concat([F11,F12,F21,F22],1),(-1,2,2))
            B = tf.matmul(F, F, transpose_b=True)
            I1 = tf.expand_dims(tf.reduce_sum(tf.linalg.diag_part(B),1),1)
            B2 = tf.matmul(B, B)
            I2 = 0.5*(I1**2 - tf.expand_dims(tf.reduce_sum(tf.linalg.diag_part(B2),1),1))
            I1r, I2r = tf.reshape(I1,(-1,1,1)), tf.reshape(I2,(-1,1,1))
            dW1 = self.C10 + self.C11*(I2r-3.0) + 2.0*self.C20*(I1r-3.0)
            dW2 = self.C01 + self.C11*(I1r-3.0) + 2*self.C02*(I2r-3.0)
            term1 = 2.0*dW1*B
            term2 = 2.0*dW2*(I1r*B - tf.matmul(B,B))
            batch = tf.shape(x)[0]
            term3 = -2.0/3.0*(dW1*I1r + 2.0*dW2*I2r)*tf.eye(2, batch_shape=[batch])
            sigma = term1 + term2 + term3
            sxx, sxy = sigma[:,0,0:1], sigma[:,0,1:2]
            syx, syy = sigma[:,1,0:1], sigma[:,1,1:2]
        sxx_x = tape.gradient(sxx, x); sxy_y = tape.gradient(sxy, y)
        syx_x = tape.gradient(syx, x); syy_y = tape.gradient(syy, y)
        fu= sxx_x + sxy_y + self.g_x
        fv= syx_x + syy_y + self.g_y
        del tape
        return fu, fv

    @tf.function
    def loss_function(self):
        u_pred, v_pred = self.net_data(self.x, self.y, self.p)
        loss_data = tf.reduce_mean((self.u - u_pred)**2) + tf.reduce_mean((self.v - v_pred)**2)
        fu, fv    = self.net_pde(self.x_f, self.y_f, self.p_f)
        loss_pde  = tf.reduce_mean(fu**2) + tf.reduce_mean(fv**2)
        loss_data= 0.1 * loss_data
        loss_pde= 10.0 * loss_pde
        
        # --- new L2 regularization term ---
        # sum of squared weights 
        l2_weights = tf.add_n([tf.nn.l2_loss(W) for W in self.nn_weights])
        reg_loss = self.lambda_reg * l2_weights

        # --- total loss ---
        total_loss = loss_data + loss_pde + reg_loss

        return loss_data, loss_pde, reg_loss, total_loss


    @tf.function
    def train_step(self):
        with tf.GradientTape() as tape:
            loss_data, loss_pde, reg_loss, total_loss = self.loss_function()
        grads = tape.gradient(total_loss, self.trainable_variables)
        self.optimizer_Adam.apply_gradients(zip(grads, self.trainable_variables))
        return loss_data, loss_pde, reg_loss, total_loss

    def get_flat_weights(self):
        return tf.concat([tf.reshape(p,[-1]) for p in self.trainable_variables],0)

    def set_flat_weights(self, flat):
        params = self.trainable_variables
        idx = 0
        for p in params:
            size = tf.size(p)
            new = tf.reshape(flat[idx:idx+size], p.shape)
            p.assign(new)
            idx += size

    def loss_and_flatgrad(self):
        with tf.GradientTape() as tape:
            # unpack the reg term too (we don’t use it here)
            _, _, _, total_loss = self.loss_function()
        grads = tape.gradient(total_loss, self.trainable_variables)
        flat_grad = tf.concat([tf.reshape(g,[-1]) for g in grads],0)
        return total_loss, flat_grad
    

    def save_history(self):
        # build the full path
        folder = os.path.join("save_loss", "BELLOW_M-R")
        # create it (and parents) if necessary
        os.makedirs(folder, exist_ok=True)

        # now save into that subfolder
        np.save(os.path.join(folder, "Data_loss.npy"),    np.array(self.data_history))
        np.save(os.path.join(folder, "PDE_loss.npy"), np.array(self.physics_history))
        np.save(os.path.join(folder, "Total_loss.npy"),   np.array(self.total_history))

        print(f"Histories saved under ./{folder}/")

    

    def train(self, nIter):

        # --- start total timer ---
        t_total_start = time.time()
        # clear old history
        self.data_history.clear()
        self.physics_history.clear()
        self.total_history.clear()

        # — Adam phase —
        t0 = time.time()
        for it in range(nIter):
            ld, lp, lr, lt = self.train_step()
            self.data_history.append(ld.numpy())
            self.physics_history.append(lp.numpy())
            self.total_history.append(lt.numpy())
            if it % 10 == 0:
                print(f"Adam Iter {it}, Data: {ld:.3e}, PDE: {lp:.3e}, Reg: {lr:.3e}, Total: {lt:.3e}")
        print(f"Adam done in {time.time()-t0:.2f}s")

        # record where Adam ended
        self.adam_steps = len(self.data_history)

        # — L-BFGS phase (unchanged) —
        def func(flat_w):
            self.set_flat_weights(tf.convert_to_tensor(flat_w, tf.float32))
            loss, grad = self.loss_and_flatgrad()
            return loss.numpy().astype(np.float64), grad.numpy().astype(np.float64)

        def cb(flat_w):
            self.set_flat_weights(tf.convert_to_tensor(flat_w, tf.float32))
            ld, lp, lr, lt = self.loss_function()
            self.data_history.append(ld.numpy())
            self.physics_history.append(lp.numpy())
            self.total_history.append(lt.numpy())
            # print each L-BFGS step
            cb.iter += 1
            print(f"L-BFGS Iter {cb.iter}, Data: {ld:.3e}, PDE: {lp:.3e}, Reg: {lr:.3e}, Total: {lt:.3e}")
        cb.iter = 0

        init_w = self.get_flat_weights().numpy()
        sopt.minimize(fun=func,
                      x0=init_w,
                      jac=True,
                      method='L-BFGS-B',
                      callback=cb,
                      options={
                          'iprint': 0,
                          'maxiter': 50000,
                          'maxfun': 50000,
                          'maxcor': 50,
                          'maxls': 50,
                          'gtol': 1e-16,
                          'ftol': 1e-16
                      })
        print("L-BFGS done")
        # --- compute + print total training time ---
        t_total = time.time() - t_total_start
        mins, secs = divmod(t_total, 60)
        print(f"Total training time: {int(mins)}m {secs:.2f}s")
        self.save_history()


    def predict(self, X_star):
        X_tf = tf.convert_to_tensor(X_star, dtype=tf.float32)
        u_pred, v_pred = self.net_data(X_tf[:,0:1], X_tf[:,1:2], X_tf[:,2:3])
        u = 0.5*(u_pred+1)*(self.u_max-self.u_min) + self.u_min
        v = 0.5*(v_pred+1)*(self.v_max-self.v_min) + self.v_min
        return u.numpy(), v.numpy()

    def plot_loss_history1(self,
                       font_family='Times New Roman',
                       title_fs=18,
                       label_fs=16,
                       tick_fs=13,
                       legend_fs=13,
                       use_log_y=True):
        import matplotlib as mpl
        import matplotlib.pyplot as plt
        import numpy as np

        # Fonts & embedding (Type 3 avoidance)
        mpl.rcParams['font.family']  = font_family
        mpl.rcParams['text.usetex']  = False
        mpl.rcParams['pdf.fonttype'] = 42
        mpl.rcParams['ps.fonttype']  = 42
        mpl.rcParams['svg.fonttype'] = 'none'

        iters = np.arange(len(self.total_history))

        fig, ax = plt.subplots(figsize=(10,6), dpi=1080)
        # Colors: Total=red, Data=blue, PDE=green
        ax.plot(iters, self.data_history,    color='blue',  linewidth=2, label='Data Loss')
        ax.plot(iters, self.physics_history, color='green', linewidth=2, label='PDE Loss')
        ax.plot(iters, self.total_history,   color='red',   linewidth=2, label='Total Loss')

        # Adam → L-BFGS switch
        ax.axvline(self.adam_steps, linestyle='--', linewidth=2, color='gray',
                   label='Adam → L-BFGS')

        if use_log_y:
            ax.set_yscale('log')

        # Labels & title with controlled fonts
        ax.set_xlabel('Iteration', fontsize=label_fs, fontname=font_family)
        ax.set_ylabel('Loss' if use_log_y else 'Loss',
                      fontsize=label_fs, fontname=font_family)
        ax.set_title('Training Loss Components', fontsize=title_fs, fontname=font_family)

        # Ticks (size + family)
        ax.tick_params(axis='both', which='both', labelsize=tick_fs)
        for tick in ax.get_xticklabels() + ax.get_yticklabels():
            tick.set_fontname(font_family)

        # X limits: start at 0, add a small right pad (≈3% of width, at least 1 step)
        n = len(self.total_history)
        right = max(0, n - 1)
        pad = max(1, int(np.ceil(0.03 * max(1, right))))  # tweak 0.03 → more/less space
        ax.set_xlim(0, right + pad)

        # Legend font control
        ax.legend(loc='upper right', frameon=False,
                  prop={'family': font_family, 'size': legend_fs})

        fig.tight_layout()
        plt.show()


def load_and_plot_loss_history(folder="save_loss/Bellow_M-R",
                               adam_steps=None,
                               font_family='Times New Roman',
                               title_fs=18,
                               label_fs=16,
                               tick_fs=13,
                               legend_fs=13,
                               use_log_y=True):
    import os
    import numpy as np
    import matplotlib as mpl
    import matplotlib.pyplot as plt

    # Fonts & embedding (Type 3 avoidance)
    mpl.rcParams['font.family']  = font_family
    mpl.rcParams['text.usetex']  = False
    mpl.rcParams['pdf.fonttype'] = 42
    mpl.rcParams['ps.fonttype']  = 42
    mpl.rcParams['svg.fonttype'] = 'none'

    # Load saved histories
    data = np.load(os.path.join(folder, "Data_loss.npy"))
    phys = np.load(os.path.join(folder, "PDE_loss.npy"))
    tot  = np.load(os.path.join(folder, "Total_loss.npy"))
    iters = np.arange(len(tot))

    fig, ax = plt.subplots(figsize=(10,6), dpi=1080)

    # Colors: Total=red, Data=blue, PDE=green
    ax.plot(iters, data, color='blue',  linewidth=2, label='Data Loss')
    ax.plot(iters, phys, color='green', linewidth=2, label='PDE Loss')
    ax.plot(iters, tot,  color='red',   linewidth=2, label='Total Loss')

    if adam_steps is not None:
        ax.axvline(adam_steps, linestyle='--', linewidth=2, color='gray',
                   label='Adam → L-BFGS')

    if use_log_y:
        ax.set_yscale('log')

    # Labels & title with controlled fonts
    ax.set_xlabel('Iteration', fontsize=label_fs, fontname=font_family)
    ax.set_ylabel('Loss' if use_log_y else 'Loss',
                  fontsize=label_fs, fontname=font_family)
    ax.set_title('Training Loss Components', fontsize=title_fs, fontname=font_family)

    # Ticks (size + family)
    ax.tick_params(axis='both', which='both', labelsize=tick_fs)
    for tick in ax.get_xticklabels() + ax.get_yticklabels():
        tick.set_fontname(font_family)

    # X limits: start at 0, end at last point
    n = len(tot)
    right = max(0, n - 1)
    pad = max(1, int(np.ceil(0.03 * max(1, right))))
    ax.set_xlim(0, right + pad)

    # Legend font control
    ax.legend(loc='upper right', frameon=False,
              prop={'family': font_family, 'size': legend_fs})

    fig.tight_layout()
    plt.show()



# =============================================================================
# Data Preparation
# =============================================================================

# Load data
data = scipy.io.loadmat('./Training_data/TRO_BELLOW_MR.mat')
p_star = data['p']
X_star = data['X_star']



# Full grid: XX, YY, TT UU, WW each shape (N, T)
XX, YY, PP = data['XX'], data['YY'], data['PP']
UU, VV = data['UU'], data['VV']

# Flatten into long vectors of length N*T
x = XX.flatten()[:,None];  y = YY.flatten()[:,None]
p = PP.flatten()[:,None];  u = UU.flatten()[:,None]
v = VV.flatten()[:,None]

# Extract initial‐condition (IC) (N,1)
x_ic = XX[:, 0 ][:,None]   
y_ic = YY[:, 0 ][:,None]
p_ic = PP[:, 0 ][:,None]
u_ic = UU[:, 0 ][:,None]
v_ic = VV[:, 0 ][:,None]



# Prepare IC training data
N_ic   = 200
X_star_ic   = np.hstack((x_ic, y_ic, p_ic))   # (N,3)
u_v_star_ic = np.hstack((u_ic, v_ic))   # (N,2)
print("X_star_ic", X_star_ic.shape)
print("u_v_star_ic", u_v_star_ic.shape)
# Sample N_ic points from the IC data
idx_ic = np.random.choice(X_star_ic.shape[0], N_ic, replace=False)
X_train_ic   = X_star_ic[idx_ic, :]    # (N_ic,3)
u_v_train_ic = u_v_star_ic[idx_ic, :]  # (N_ic,2)
print("X_train_ic", X_train_ic.shape)
print("u_v_train_ic", u_v_train_ic.shape)




# Preparing supervised training data
N_u = 5000
N, T = X_star.shape[0], p_star.shape[0]
idx = np.random.choice(N*T, N_u, replace=False)
X_data_train = np.hstack((x[idx], y[idx], p[idx]))
u_v_train = np.hstack((u[idx], v[idx]))
# Combine IC samples with your original supervised samples ---
X_data_train = np.vstack((X_data_train, X_train_ic))  # (N_train+N_ic, 3)
u_v_train    = np.vstack((u_v_train, u_v_train_ic))  # (N_train+N_ic, 2)
print("X_data_train", X_data_train.shape)
print("u_v_train", u_v_train.shape)



# Preparing unsupervised training data (collocation points)
N_f = 5000
X_star_total = np.hstack((x, y, p))
lb_bound = X_star_total.min(0)
ub_bound = X_star_total.max(0)
print("X_star_total", X_star_total.shape)
X_f_train = lb_bound + (ub_bound - lb_bound)*lhs(3, N_f)
X_f_train = np.vstack((X_f_train, X_data_train))
print("X_f_train", X_f_train.shape)

g_x, g_y = 0, -0.000012 # body force term gy= 1.1772e-5 
C10, C01, C20, C11, C02 = -0.233, 2.562, 0.116, -0.561, 0.9
layers = [3,30,30,30,30,30,30,30,2]

# =============================================================================
# Train
# =============================================================================

model = PhysicsInformedNN(X_data_train, u_v_train, X_f_train,
                          lb_bound, ub_bound,
                          C10, C01, C20, C11, C02,
                          g_x, g_y, layers)

model.train(1000)

TensorFlow version: 2.10.1
Numpy version: 1.23.1
scipy version: 1.13.1
matplotlib version: 3.9.2
X_star_ic (20752, 3)
u_v_star_ic (20752, 2)
X_train_ic (200, 3)
u_v_train_ic (200, 2)
X_data_train (5200, 3)
u_v_train (5200, 2)
X_star_total (954592, 3)
X_f_train (10200, 3)
Adam Iter 0, Data: 9.786e-02, PDE: 7.904e-08, Reg: 7.485e-07, Total: 9.786e-02
Adam Iter 10, Data: 1.288e-02, PDE: 4.422e-06, Reg: 7.539e-07, Total: 1.289e-02
Adam Iter 20, Data: 6.762e-03, PDE: 4.278e-06, Reg: 7.530e-07, Total: 6.767e-03
Adam Iter 30, Data: 2.959e-03, PDE: 4.082e-06, Reg: 7.523e-07, Total: 2.963e-03
Adam Iter 40, Data: 1.716e-03, PDE: 4.751e-06, Reg: 7.529e-07, Total: 1.722e-03
Adam Iter 50, Data: 1.656e-03, PDE: 5.932e-06, Reg: 7.531e-07, Total: 1.662e-03
Adam Iter 60, Data: 1.498e-03, PDE: 6.107e-06, Reg: 7.524e-07, Total: 1.505e-03
Adam Iter 70, Data: 1.428e-03, PDE: 6.892e-06, Reg: 7.523e-07, Total: 1.436e-03
Adam Iter 80, Data: 1.384e-03, PDE: 6.758e-06, Reg: 7.520e-07, Total: 1.391e-03
Adam Iter

# Loss curve plotting

In [2]:
model.plot_loss_history1(title_fs=20, label_fs=18, tick_fs=14, legend_fs=14)

# load the saved loss to check

In [3]:
# Loader with different sizes
load_and_plot_loss_history(adam_steps=model.adam_steps, title_fs=20, label_fs=18, tick_fs=14, legend_fs=14)

# Total loss

In [4]:
x1 = XX.flatten()[:,None];  y1 = YY.flatten()[:,None]
p1 = PP.flatten()[:,None];  u1 = UU.flatten()[:,None]
v1 = VV.flatten()[:,None]

# Prediction
starr = np.hstack((x1, y1, p1))
u_pred, v_pred = model.predict(starr)


# Error
error_u = np.linalg.norm(u1-u_pred,2)/np.linalg.norm(u1,2)
error_v = np.linalg.norm(v1-v_pred,2)/np.linalg.norm(v1,2)
print('Total Error u: %e' % (error_u)) 
print('Total Error v: %e' % (error_v))

Total Error u: 3.528191e-03
Total Error v: 9.014502e-03


# Different pressures error

In [5]:
snap = np.array([9])

x_star = XX[:,snap]
y_star = YY[:,snap]
p_star = PP[:,snap]
u_star = UU[:,snap]
v_star = VV[:,snap]


# Prediction
starr = np.hstack((x_star, y_star, p_star))
u_pred, v_pred = model.predict(starr)
# Error
error_u = np.linalg.norm(u_star-u_pred,2)/np.linalg.norm(u_star,2)
error_v = np.linalg.norm(v_star-v_pred,2)/np.linalg.norm(v_star,2)

print('Error u at 30 kPa: %e' % (error_u)) 
print('Error v at 30 kPa: %e' % (error_v))    
#-----------------------------------------
snap = np.array([27])
x_star = XX[:,snap]
y_star = YY[:,snap]
p_star = PP[:,snap]
u_star = UU[:,snap]
v_star = VV[:,snap]


# Prediction
starr = np.hstack((x_star, y_star, p_star))
u_pred, v_pred = model.predict(starr)
# Error
error_u = np.linalg.norm(u_star-u_pred,2)/np.linalg.norm(u_star,2)
error_v = np.linalg.norm(v_star-v_pred,2)/np.linalg.norm(v_star,2)

print('Error u at 90 kPa: %e' % (error_u)) 
print('Error v at 90 kPa: %e' % (error_v))    
#------------------------------------------
snap = np.array([36])
x_star = XX[:,snap]
y_star = YY[:,snap]
p_star = PP[:,snap]
u_star = UU[:,snap]
v_star = VV[:,snap]

# Prediction
starr = np.hstack((x_star, y_star, p_star))
u_pred, v_pred = model.predict(starr)
# Error
error_u = np.linalg.norm(u_star-u_pred,2)/np.linalg.norm(u_star,2)
error_v = np.linalg.norm(v_star-v_pred,2)/np.linalg.norm(v_star,2)

print('Error u at 120 kPa: %e' % (error_u)) 
print('Error v at 120 kPa: %e' % (error_v))    
#------------------------------------------
snap = np.array([45])
x_star = XX[:,snap]
y_star = YY[:,snap]
p_star = PP[:,snap]
u_star = UU[:,snap]
v_star = VV[:,snap]


# Prediction
starr = np.hstack((x_star, y_star, p_star))
u_pred, v_pred = model.predict(starr)
# Error
error_u = np.linalg.norm(u_star-u_pred,2)/np.linalg.norm(u_star,2)
error_v = np.linalg.norm(v_star-v_pred,2)/np.linalg.norm(v_star,2)

print('Error u at 150 kPa: %e' % (error_u)) 
print('Error v at 150 kPa: %e' % (error_v))    

Error u at 30 kPa: 3.698360e-03
Error v at 30 kPa: 3.444102e-02
Error u at 90 kPa: 3.916714e-03
Error v at 90 kPa: 1.067611e-02
Error u at 120 kPa: 3.131388e-03
Error v at 120 kPa: 8.493348e-03
Error u at 150 kPa: 2.901924e-03
Error v at 150 kPa: 6.853023e-03


# Plot and save deformed exact shape vs predicted shape at 100 kPa snap 30

In [ ]:
import os
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize

# ---- Fonts (Times-like) + avoid Type 3 in PDF/PS ----
mpl.rcParams.update({
    'font.family': ['Times New Roman', 'STIXGeneral', 'DejaVu Serif'],
    'mathtext.fontset': 'stix',
    'text.usetex': False,
    'pdf.fonttype': 42,
    'ps.fonttype': 42,
    'svg.fonttype': 'none',
})

# ===== USER SETTINGS =====
SNAP      = 30                 # which snapshot to plot
XLIM      = (-120.0, 10.0)      # control whitespace on X axis
CMAP      = "jet"
PNG_DPI   = 1080
PARENT    = "save_plots"
SUB       = "BELLOW_MR"
# =========================

# ---- Data for the chosen snap (expects XX,YY,PP,UU,VV,X_star,model in memory) ----
x_star = XX[:, [SNAP]]
y_star = YY[:, [SNAP]]
p_star = PP[:, [SNAP]]
u_star = UU[:, [SNAP]]
v_star = VV[:, [SNAP]]

# undeformed shape from X_star (col 0 = reference/IC)
x_ic = X_star[:, 0:1].astype(np.float32)
y_ic = X_star[:, 1:2].astype(np.float32)

# prediction
starr = np.hstack((x_star, y_star, p_star))
u_pred, v_pred = model.predict(starr)  # returns numpy arrays

# shared color limits (same for GT & Pred)
umin_u = float(min(u_star.min(), u_pred.min()))
umax_u = float(max(u_star.max(), u_pred.max()))
vmin_v = float(min(v_star.min(), v_pred.min()))
vmax_v = float(max(v_star.max(), v_pred.max()))
norm_u = Normalize(vmin=umin_u, vmax=umax_u)
norm_v = Normalize(vmin=vmin_v, vmax=vmax_v)

# plotting helper
def plot_deformation(x, y, u, v, comp_label, norm, component='u',
                     cmap='jet', xlim=None, textbox_xy=(0.035, 0.05),
                     title=None, save_stub=None,
                     # --- new font controls ---
                     cbar_label_fs=28, cbar_tick_fs=18, box_fs=21,
                     label_fs=28, tick_fs=18, title_fs=20):
    x = x.ravel(); y = y.ravel()
    u = u.ravel(); v = v.ravel()

    x_def = -x - u
    y_def =  y + v

    if component == 'u':
        cfield = u
        label_text = f"Max {comp_label}: {u.max():.2f}\nMin {comp_label}: {u.min():.2f}"
    else:
        cfield = v
        label_text = f"Max {comp_label}: {v.max():.2f}\nMin {comp_label}: {v.min():.2f}"

    fig, ax = plt.subplots(figsize=(12, 6), dpi=1080)
    ax.scatter(-x, y, color='grey', alpha=1, s=0.009)
    ax.scatter(x_def, y_def, c=cfield, cmap=cmap, norm=norm, s=1, alpha=1)

    # Colorbar font sizes
    cbar = plt.colorbar(plt.cm.ScalarMappable(cmap=cmap, norm=norm), ax=ax)
    # cbar.set_label('Deformation', fontsize=cbar_label_fs)
    cbar.ax.tick_params(labelsize=cbar_tick_fs)

    # Max/Min box font size
    ax.text(textbox_xy[0], textbox_xy[1], label_text, transform=ax.transAxes,
            fontsize=box_fs, va='bottom',
            bbox=dict(boxstyle='round,pad=0.5', facecolor='white', alpha=0.5))

    # Axes labels / ticks / title sizes
    ax.set_xlabel('X Coordinate', fontsize=label_fs)
    ax.set_ylabel('Y Coordinate', fontsize=label_fs)
    ax.tick_params(axis='both', labelsize=tick_fs)
    ax.set_aspect('equal', adjustable='box')

    if xlim is not None:
        ax.set_xlim(*xlim)
    if title:
        ax.set_title(title, fontsize=title_fs)

    fig.tight_layout()
    if save_stub:
        os.makedirs(os.path.join(PARENT, SUB), exist_ok=True)
        base = os.path.join(PARENT, SUB, save_stub)
        plt.savefig(base + ".pdf", bbox_inches='tight')
        plt.savefig(base + ".png", dpi=PNG_DPI, bbox_inches='tight')
    plt.show()

# U component (u_x): Exact and prediction
label_ux     = r"$u_x$"
label_ux_hat = r"$\hat{u}_x$"

plot_deformation(x_ic, y_ic, u_star, v_star, label_ux,
                 norm_u, component='u', cmap=CMAP, xlim=XLIM,
                 title=rf"BELLOW_MR_EXACT u_x at snap {SNAP}",   save_stub=f"BELLOW_MR_ux_EXACT_snap{SNAP}",
                 cbar_label_fs=30, cbar_tick_fs=20, box_fs=22)

plot_deformation(x_ic, y_ic, u_pred, v_pred, label_ux_hat,
                 norm_u, component='u', cmap=CMAP, xlim=XLIM,
                 title=rf"BELLOW_MR_Prediction u_x at snap {SNAP}", save_stub=f"BELLOW_MR_ux_PRED_snap{SNAP}",
                 cbar_label_fs=30, cbar_tick_fs=20, box_fs=22)

# V component (u_y): Exact and prediction
label_uy     = r"$u_y$"
label_uy_hat = r"$\hat{u}_y$"

plot_deformation(x_ic, y_ic, u_star, v_star, label_uy, norm_v, component='v',
                 cmap=CMAP, xlim=XLIM, title=rf"BELLOW_MR_EXACT u_y at snap {SNAP}", save_stub=f"BELLOW_MR_uy_EXACT_snap{SNAP}",
                 cbar_label_fs=30, cbar_tick_fs=20, box_fs=22)

plot_deformation(x_ic, y_ic, u_pred, v_pred, label_uy_hat,
                 norm_v, component='v', cmap=CMAP, xlim=XLIM,
                 title=rf"BELLOW_MR_Prediction u_y at snap {SNAP}", save_stub=f"BELLOW_MR_uy_PRED_snap{SNAP}",
                 cbar_label_fs=30, cbar_tick_fs=20, box_fs=22)

# ---- Save a compact bundle with everything needed to re-plot later ----
bundle = {
    "snap":        np.int32(SNAP),
    "x_ic":        x_ic.astype(np.float32),
    "y_ic":        y_ic.astype(np.float32),
    "u_star":      u_star.astype(np.float32),
    "v_star":      v_star.astype(np.float32),
    "u_pred":      u_pred.astype(np.float32),
    "v_pred":      v_pred.astype(np.float32),
    "xlim":        np.array(XLIM, dtype=np.float32),
    "cmap":        CMAP,
    "label_ux":    label_ux,
    "label_ux_hat":label_ux_hat,
    "label_uy":    label_uy,
    "label_uy_hat":label_uy_hat,
    "umin_u":      np.float32(umin_u),
    "umax_u":      np.float32(umax_u),
    "vmin_v":      np.float32(vmin_v),
    "vmax_v":      np.float32(vmax_v),
}
os.makedirs(os.path.join(PARENT, SUB), exist_ok=True)
bundle_path = os.path.join(PARENT, SUB, f"BELLOW_MR_snap{SNAP}.npz")
np.savez_compressed(bundle_path, **bundle)
print("Saved bundle:", os.path.abspath(bundle_path))
